# 4-Backend Object Detection Benchmark on COCO15

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/integration/Multi_Backend_COCO15.ipynb)

This notebook evaluates **four distinct deep-learning backends** side by side through Klygo's unified `models.Detector` interface on the real 128-image COCO15 dataset (`artifacts/integration/coco15`):

1. **Ultralytics YOLOv8n** (`yolov8n.pt`, Backend: `"Ultralytics"`)
2. **Ultralytics YOLO-World v2** (`yolov8s-worldv2.pt`, Backend: `"Ultralytics"`)
3. **Torchvision Faster R-CNN** (`fasterrcnn_mobilenet_v3_large_fpn`, Backend: `"Torchvision"`)
4. **KerasHub RT-DETR / D-FINE** (`dfine_small_coco`, Backend: `"Keras"` on `KERAS_BACKEND=torch`)


In [ ]:
import os
os.environ.setdefault('KERAS_BACKEND', 'torch')

import importlib.util, subprocess, sys
if importlib.util.find_spec('klygo') is None:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--upgrade', '--no-cache-dir',
        'git+https://github.com/IchigoMazone/klygo.git@v2-release'
    ])
for pkg in ('torch', 'torchvision', 'ultralytics', 'keras', 'keras_hub'):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg.replace('_', '-')])


In [ ]:
from klygo import config, datasets, files, media, models, outputs, postprocessing, preprocessing, utils, visual
from examples.integration.multi_backend_workflow import (
    LocalYOLODetector,
    LocalYOLOWorldDetector,
    LocalTorchvisionDetector,
    LocalKerasDetector,
    run_all,
)

dataset_root = files.path('artifacts/integration/coco15')
if not files.is_file(files.join(dataset_root, 'data.yaml')):
    from test.integration.build_coco15 import build_coco15
    build_coco15(dataset_root, download=True)

info = datasets.info(dataset_root, verbose=False)
print('Dataset images:', info['total_images'], '| Classes:', len(info['classes']))


## Jump to an API

- [`models.Detector`](#modelsdetector): unified multi-backend detection contract (`cast_inputs`, `run_inference`, `format_results`).
- [`outputs.build_detections`](#outputsbuild_detections): pack normalized predictions into `Detection` frames.
- [`postprocessing.nms` & `postprocessing.box_iou`](#postprocessingsuppress--postprocessingbox_iou): backend-neutral NMS operation and ground-truth IoU matching.


### models.Detector

Each backend wrapper subclasses `models.Detector` and delegates tensor casting, forward inference, and output formatting to `klygo.models.backend.common`:
- `LocalYOLODetector` & `LocalYOLOWorldDetector` (`backend="Ultralytics"`)
- `LocalTorchvisionDetector` (`backend="Torchvision"` — `fasterrcnn_mobilenet_v3_large_fpn`)
- `LocalKerasDetector` (`backend="Keras"` — `keras_hub.models.ObjectDetector.from_preset("dfine_small_coco")`)


In [ ]:
for cls in (LocalYOLODetector, LocalYOLOWorldDetector, LocalTorchvisionDetector, LocalKerasDetector):
    print(cls.__name__, '-> issubclass(Detector):', issubclass(cls, models.Detector))


### outputs.build_detections

### postprocessing.nms & postprocessing.box_iou

## Put it together

Run the full 4-backend evaluation across COCO15, computing per-class and aggregate TP, FP, FN, Precision, Recall, F1-Score, and latency/FPS, and writing annotated preview images and the JSON report under `artifacts/integration/four-backends-eval/`.


In [ ]:
run_all()


In [ ]:
report_path = files.path('artifacts/integration/four-backends-eval/report/multi_backend_report.json')
report = files.load(report_path, verbose=False)
for m in report['models']:
    t = m['totals']
    b = m['benchmark']
    print(
        f"{m['model']} ({m['backend']}): "
        f"P={t['precision']:.3f}, R={t['recall']:.3f}, F1={t['f1']:.3f} | "
        f"{b['avg_latency_ms']:.1f} ms ({b['fps']:.1f} FPS)"
    )


## Remember

- `models.Detector` normalizes predictions from `Ultralytics`, `Hugging Face`, `Torchvision`, and `Keras` into identical `klygo.outputs.Detections` containers.
- `klygo.models.backend.keras` natively supports `KerasHub` `ObjectDetector` presets (such as D-FINE / RT-DETR) on `KERAS_BACKEND=torch` without requiring TensorFlow.
- Explore the [Models Overview](../models/Document_Models_Overview.ipynb) ([Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/models/Document_Models_Overview.ipynb)), [Backend Dispatchers Guide](../../docs/models/backend/README.md), and [4-Backend Workflow Guide](../../docs/integration/multi-backend-workflow.md).
